## scripts to clean raw election data

In [2]:
import pandas as pd


In [3]:
#read in data, skipping the first several rows which contain a lot of blank cells and formatting
HoC_7_20_2025 = pd.read_excel("../data/raw/election_results/July 20 2025 House of Councillors election results.xlsx", skiprows = 7)

In [4]:
HoC_7_20_2025.head()

,Unnamed: 0,Unnamed: 1,得票率（％）,得票総数,を除く）の得票総数,Unnamed: 5,得票率（％）.1,得票総数.1,を除く）の得票総数.1,Unnamed: 9,得票率（％）.2,得票総数.2,を除く）の得票総数.2,Unnamed: 13,得票率（％）.3,得票総数.3,を除く）の得票総数.3
0,北海道,"624,976.406",24.59,"380,784.230","244,192.176","441,997.778",17.39,"356,390.310","85,607.468","75,510.486",2.97,"69,218","6,292.486","219,288.596",8.63,"97,846","121,442.596"
1,青森県,"156,328.405",29.01,"111,841","44,487.405","104,349.326",19.36,"89,879.723","14,469.603","11,799.500",2.19,"10,733","1,066.500","40,124.826",7.45,"19,192","20,932.826"
2,岩手県,"134,061.347",23.69,"99,391","34,670.347","128,614.132",22.73,"108,350.112","20,264.020","16,377.216",2.89,"15,058","1,319.216","35,460.818",6.27,"18,856","16,604.818"
3,宮城県,"250,727.201",24.46,"168,904.054","81,823.147","167,213.819",16.31,"142,056.166","25,157.653","37,963.077",3.70,"34,044","3,919.077","81,381.609",7.94,"35,637","45,744.609"
4,秋田県,"157,492.272",35.44,"119,637","37,855.272","60,813.866",13.69,"48,443.135","12,370.731","14,474.314",3.26,"13,544",930.314,"32,189.004",7.24,"15,519","16,670.004"


In [5]:
#only select columns for prefecture and 得票総数 for each party
vote_totals_only = HoC_7_20_2025.loc[:, ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 5', 'Unnamed: 9', 'Unnamed: 13']]

In [6]:
vote_totals_only.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 5,Unnamed: 9,Unnamed: 13
0,北海道,"624,976.406","441,997.778","75,510.486","219,288.596"
1,青森県,"156,328.405","104,349.326","11,799.500","40,124.826"
2,岩手県,"134,061.347","128,614.132","16,377.216","35,460.818"
3,宮城県,"250,727.201","167,213.819","37,963.077","81,381.609"
4,秋田県,"157,492.272","60,813.866","14,474.314","32,189.004"


In [7]:
#because the original excel is split on multiple pages, need to access each page 
#and then merge them by prefecture
#page 5 contains the total votes by prefecture; only select the first two columns

page_1 = vote_totals_only.loc[:46, :]
page_2 = vote_totals_only.loc[54:100, :]
page_3 = vote_totals_only.loc[108:154, :]
page_4 = vote_totals_only.loc[162:208, :]
page_5 = vote_totals_only.loc[216:262, ['Unnamed: 0', 'Unnamed: 1']]

In [8]:
#rename columns to comprehensible labels; all columns are labeled by political party, 
#the value indicates votes for that party in a particular prefecture
page_1.columns = ['prefecture', '自由民主党', '立憲民主党', '日本維新の会', '公明党']
page_2.columns = ['prefecture', '国民民主党', '日本共産党', 'れいわ新撰組', '参政等']
page_3.columns = ['prefecture', '日本保守党', '社会民主党', '無所属連合', 'チームみらい']
page_4.columns = ['prefecture', '日本誠心会', '日本改革党', '再生の道', 'NHK党']
page_5.columns = ['prefecture', 'total']

In [9]:
#merge renamed page dataframes based on prefecture
merged_vote_totals = pd.merge(page_1, page_2, on = 'prefecture')
merged_vote_totals = pd.merge(merged_vote_totals, page_3, on = 'prefecture')
merged_vote_totals = pd.merge(merged_vote_totals, page_4, on = 'prefecture')
merged_vote_totals = pd.merge(merged_vote_totals, page_5, on = 'prefecture')

In [10]:
merged_vote_totals.tail()

,prefecture,自由民主党,立憲民主党,日本維新の会,公明党,国民民主党,日本共産党,れいわ新撰組,参政等,日本保守党,社会民主党,無所属連合,チームみらい,日本誠心会,日本改革党,再生の道,NHK党,total
42,熊本県,"242,862.712","112,899.172","31,722.707","80,303.263","69,967.736","20,075.550","53,093.217","119,194.807","32,817","16,769.720","3,334.029","9,370.860","4,280",533,"6,554.761","7,597.243","811,375.777"
43,大分県,"131,327.741","95,251.070","19,126.038","53,822.591","49,673.681","17,739.905","35,471.721","64,748.095","21,933","19,368.137","1,518.064","5,557.039","3,672",345,"4,040.380","5,521.430","529,115.892"
44,宮崎県,"135,201.367","70,339.734","17,200.113","52,121.768","45,324.780","12,962.886","35,060.772","66,717.020","18,495","15,044.119","1,309.045","4,032","2,658",343,"3,343.991","4,141.294","484,294.889"
45,鹿児島県,"226,583.045","88,081.748","29,274.417","65,341.562","55,394.928","18,921.217","52,722.021","100,949.241","27,458","17,736.439","3,375.005","7,873.714","3,552",443,"5,970.386","6,792.093","710,468.816"
46,沖縄県,"107,120.107","61,917.899","36,601.778","67,344.982","57,059.762","36,313.566","65,588.459","81,473.722","23,832.448","60,453.461","5,084.249","10,052.558","4,704",532,"7,094.930","7,738.679","632,912.600"


In [11]:
#remove commas from numeric data
merged_vote_totals = merged_vote_totals.replace(',', '', regex = True)

In [12]:
#convert str vote data to float
for column in merged_vote_totals.columns[1:]:
    merged_vote_totals[[column]] = merged_vote_totals[[column]].astype(float)

In [13]:
merged_vote_totals.head()

,prefecture,自由民主党,立憲民主党,日本維新の会,公明党,国民民主党,日本共産党,れいわ新撰組,参政等,日本保守党,社会民主党,無所属連合,チームみらい,日本誠心会,日本改革党,再生の道,NHK党,total
0,北海道,624976.406,441997.778,75510.486,219288.596,291341.384,147611.256,189562.505,275373.672,121945.913,47154.219,9997.166,41605.003,11459.0,2777.0,16928.422,24360.552,2541889.358
1,青森県,156328.405,104349.326,11799.500,40124.826,52545.528,29338.788,39599.431,61157.796,18406.769,9071.594,1625.085,4699.757,1851.0,373.0,3027.047,4634.977,538932.829
2,岩手県,134061.347,128614.132,16377.216,35460.818,62302.749,27494.691,42952.994,65301.990,18376.276,14510.755,1561.031,6783.283,1994.0,397.0,3674.387,5989.187,565851.856
3,宮城県,250727.201,167213.819,37963.077,81381.609,117161.995,45026.771,86070.035,115396.317,47552.777,20992.837,3268.086,29201.430,4451.0,1059.0,7509.259,10213.547,1025188.760
4,秋田県,157492.272,60813.866,14474.314,32189.004,53440.108,15863.527,26542.705,41725.234,16136.000,12229.801,1733.120,3879.914,1223.0,283.0,2471.032,3849.958,444346.855


In [14]:
#create separate dataframe for vote percentage
merged_vote_percentage = merged_vote_totals.iloc[:, 0:17].copy()

In [15]:
merged_vote_percentage.tail()

,prefecture,自由民主党,立憲民主党,日本維新の会,公明党,国民民主党,日本共産党,れいわ新撰組,参政等,日本保守党,社会民主党,無所属連合,チームみらい,日本誠心会,日本改革党,再生の道,NHK党
42,熊本県,242862.712,112899.172,31722.707,80303.263,69967.736,20075.550,53093.217,119194.807,32817.000,16769.720,3334.029,9370.860,4280.0,533.0,6554.761,7597.243
43,大分県,131327.741,95251.070,19126.038,53822.591,49673.681,17739.905,35471.721,64748.095,21933.000,19368.137,1518.064,5557.039,3672.0,345.0,4040.380,5521.430
44,宮崎県,135201.367,70339.734,17200.113,52121.768,45324.780,12962.886,35060.772,66717.020,18495.000,15044.119,1309.045,4032.000,2658.0,343.0,3343.991,4141.294
45,鹿児島県,226583.045,88081.748,29274.417,65341.562,55394.928,18921.217,52722.021,100949.241,27458.000,17736.439,3375.005,7873.714,3552.0,443.0,5970.386,6792.093
46,沖縄県,107120.107,61917.899,36601.778,67344.982,57059.762,36313.566,65588.459,81473.722,23832.448,60453.461,5084.249,10052.558,4704.0,532.0,7094.930,7738.679


In [16]:
#calculate percentage by dividing through by vote total
merged_vote_percentage.iloc[:, 1:17] = merged_vote_percentage.iloc[:, 1:17].div(merged_vote_totals['total'], axis = 0)

In [17]:
merged_vote_totals.set_index('prefecture', inplace = True)
merged_vote_percentage.set_index('prefecture', inplace = True)

In [18]:
merged_vote_percentage.head(2)

,自由民主党,立憲民主党,日本維新の会,公明党,国民民主党,日本共産党,れいわ新撰組,参政等,日本保守党,社会民主党,無所属連合,チームみらい,日本誠心会,日本改革党,再生の道,NHK党
prefecture,,,,,,,,,,,,,,,,
北海道,0.245871,0.173886,0.029706,0.086270,0.114616,0.058071,0.074575,0.108334,0.047975,0.018551,0.003933,0.016368,0.004508,0.001092,0.006660,0.009584
青森県,0.290070,0.193622,0.021894,0.074452,0.097499,0.054439,0.073477,0.113479,0.034154,0.016833,0.003015,0.008720,0.003435,0.000692,0.005617,0.008600


In [21]:
japanese_parties = ['自由民主党', '立憲民主党', '日本維新の会', '公明党', 
                    '国民民主党', '日本共産党', 'れいわ新撰組', '参政党',
       '日本保守党', '社会民主党', '無所属連合', 'チームみらい', '日本誠心会', 
       '日本改革党', '再生の道', 'NHK党', '幸福実現党', 'ごぼうの党', '日本第一党',
       '新党くにもり', '維新政党・新風']
english_parties = ['Liberal Democratic Party', 'Constitutional Democratic Party', 'Japan Innovation Party', 'Komeito', 
                   'Democratic Party For the People', 'Japanese Communist Party', 'Reiwa Shinsengumi', 
                   'Sanseito', 'Conservative Party of Japan', 'Social Democratic Party', 
                   'Independent Alliance', 'Team Mirai', 'Nippon Seishinkai', 'Nihon Kakumeito', 
                   'Saisei no Michi', 'NHK Party', 'The Happiness Realization Party', 'Gobonoto', 'Japan First Party',
                    'Shinto Kunimori', 'Ishin Party Shimpu' ]
abbreviations = ['LDP', 'CDP', 'JIP', 'NKP', 'DPP', 'JCP', 'REIWA', 'DIY', 'CPJ', 'SDP', 'IND', 
                 'MIRAI', 'NSK', 'NKT', 'SAISEI', 'NHK', 'HRP', 'GBT', 'JFP', 'STK', 'IPS']
english_prefectures = ['Hokkaido', 'Aomori', 'Iwate', 'Miyagi', 'Akita', 'Yamagata', 'Fukushima', 'Ibaraki',
                       'Tochigi', 'Gunma', 'Saitama', 'Chiba', 'Tokyo', 'Kanagawa', 'Niigata',
                       'Toyama', 'Ishikawa', 'Fukui', 'Yamanashi', 'Nagano', 'Gifu', 'Shizuoka', 'Aichi',
                       'Mie', 'Shiga', 'Kyoto', 'Osaka', 'Hyogo', 'Nara', 'Wakayama', 'Tottori', 'Shimane', 'Okayama',
                       'Hiroshima', 'Yamaguchi', 'Tokushima', 'Kagawa', 'Ehime', 'Kochi', 'Fukuoka', 
                       'Saga', 'Nagasaki', 'Kumamoto', 'Oita', 'Miyazaki', 'Kagoshima', 'Okinawa']
japanese_prefectures = ['北海道', '青森県', '岩手県', '宮城県', '秋田県', '山形県', '福島県', '茨城県', '栃木県', '群馬県',
       '埼玉県', '千葉県', '東京都', '神奈川県', '新潟県', '富山県', '石川県', '福井県', '山梨県', '長野県',
       '岐阜県', '静岡県', '愛知県', '三重県', '滋賀県', '京都府', '大阪府', '兵庫県', '奈良県', '和歌山県',
       '鳥取県', '島根県', '岡山県', '広島県', '山口県', '徳島県', '香川県', '愛媛県', '高知県', '福岡県',
       '佐賀県', '長崎県', '熊本県', '大分県', '宮崎県', '鹿児島県', '沖縄県']
jp_prefectures = list(merged_vote_percentage.index)
jp_to_eng = dict(zip(japanese_parties, english_parties))
jp_to_abbrv = dict(zip(japanese_parties, abbreviations))
eng_to_jp = dict(zip(english_parties, japanese_parties))
eng_to_abbrv = dict(zip(english_parties, abbreviations))
jp_prefect_to_eng = dict(zip(merged_vote_percentage.index, english_prefectures))
eng_to_jp_prefect = dict(zip(english_prefectures, merged_vote_percentage.index))

In [29]:
eng_to_jp_prefect

{'Hokkaido': '北海道',
 'Aomori': '青森県',
 'Iwate': '岩手県',
 'Miyagi': '宮城県',
 'Akita': '秋田県',
 'Yamagata': '山形県',
 'Fukushima': '福島県',
 'Ibaraki': '茨城県',
 'Tochigi': '栃木県',
 'Gunma': '群馬県',
 'Saitama': '埼玉県',
 'Chiba': '千葉県',
 'Tokyo': '東京都',
 'Kanagawa': '神奈川県',
 'Niigata': '新潟県',
 'Toyama': '富山県',
 'Ishikawa': '石川県',
 'Fukui': '福井県',
 'Yamanashi': '山梨県',
 'Nagano': '長野県',
 'Gifu': '岐阜県',
 'Shizuoka': '静岡県',
 'Aichi': '愛知県',
 'Mie': '三重県',
 'Shiga': '滋賀県',
 'Kyoto': '京都府',
 'Osaka': '大阪府',
 'Hyogo': '兵庫県',
 'Nara': '奈良県',
 'Wakayama': '和歌山県',
 'Tottori': '鳥取県',
 'Shimane': '島根県',
 'Okayama': '岡山県',
 'Hiroshima': '広島県',
 'Yamaguchi': '山口県',
 'Tokushima': '徳島県',
 'Kagawa': '香川県',
 'Ehime': '愛媛県',
 'Kochi': '高知県',
 'Fukuoka': '福岡県',
 'Saga': '佐賀県',
 'Nagasaki': '長崎県',
 'Kumamoto': '熊本県',
 'Oita': '大分県',
 'Miyazaki': '宮崎県',
 'Kagoshima': '鹿児島県',
 'Okinawa': '沖縄県'}

In [19]:
merged_vote_percentage.index

Index(['北海道', '青森県', '岩手県', '宮城県', '秋田県', '山形県', '福島県', '茨城県', '栃木県', '群馬県',
       '埼玉県', '千葉県', '東京都', '神奈川県', '新潟県', '富山県', '石川県', '福井県', '山梨県', '長野県',
       '岐阜県', '静岡県', '愛知県', '三重県', '滋賀県', '京都府', '大阪府', '兵庫県', '奈良県', '和歌山県',
       '鳥取県', '島根県', '岡山県', '広島県', '山口県', '徳島県', '香川県', '愛媛県', '高知県', '福岡県',
       '佐賀県', '長崎県', '熊本県', '大分県', '宮崎県', '鹿児島県', '沖縄県'],
      dtype='object', name='prefecture')

In [164]:
#copy dataframes to create an english version
eng_vote_totals = merged_vote_totals.copy()
eng_vote_percent = merged_vote_percentage.copy()

In [167]:
eng_vote_totals.index = english_prefectures
eng_vote_totals = eng_vote_totals.rename(columns=jp_to_eng)

eng_vote_percent.index = english_prefectures
eng_vote_percent = eng_vote_percent.rename(columns=jp_to_eng)

In [168]:
eng_vote_totals.head()

,Liberal Democratic Party,Constitutional Democratic Party,Japan Innovation Party,Komeito,Democratic Party For the People,Japanese Communist Party,Reiwa Shinsengumi,Sanseito,Conservative Party of Japan,Social Democratic Party,Independent Alliance,Team Mirai,Nippon Seishinkai,Nihon Kakumeito,Saisei no Michi,NHK Party,total
Hokkaido,624976.406,441997.778,75510.486,219288.596,291341.384,147611.256,189562.505,275373.672,121945.913,47154.219,9997.166,41605.003,11459.0,2777.0,16928.422,24360.552,2541889.358
Aomori,156328.405,104349.326,11799.500,40124.826,52545.528,29338.788,39599.431,61157.796,18406.769,9071.594,1625.085,4699.757,1851.0,373.0,3027.047,4634.977,538932.829
Iwate,134061.347,128614.132,16377.216,35460.818,62302.749,27494.691,42952.994,65301.990,18376.276,14510.755,1561.031,6783.283,1994.0,397.0,3674.387,5989.187,565851.856
Miyagi,250727.201,167213.819,37963.077,81381.609,117161.995,45026.771,86070.035,115396.317,47552.777,20992.837,3268.086,29201.430,4451.0,1059.0,7509.259,10213.547,1025188.760
Akita,157492.272,60813.866,14474.314,32189.004,53440.108,15863.527,26542.705,41725.234,16136.000,12229.801,1733.120,3879.914,1223.0,283.0,2471.032,3849.958,444346.855


In [169]:
eng_vote_percent.head()

,Liberal Democratic Party,Constitutional Democratic Party,Japan Innovation Party,Komeito,Democratic Party For the People,Japanese Communist Party,Reiwa Shinsengumi,Sanseito,Conservative Party of Japan,Social Democratic Party,Independent Alliance,Team Mirai,Nippon Seishinkai,Nihon Kakumeito,Saisei no Michi,NHK Party
Hokkaido,0.245871,0.173886,0.029706,0.086270,0.114616,0.058071,0.074575,0.108334,0.047975,0.018551,0.003933,0.016368,0.004508,0.001092,0.006660,0.009584
Aomori,0.290070,0.193622,0.021894,0.074452,0.097499,0.054439,0.073477,0.113479,0.034154,0.016833,0.003015,0.008720,0.003435,0.000692,0.005617,0.008600
Iwate,0.236920,0.227293,0.028943,0.062668,0.110104,0.048590,0.075909,0.115405,0.032475,0.025644,0.002759,0.011988,0.003524,0.000702,0.006494,0.010584
Miyagi,0.244567,0.163105,0.037030,0.079382,0.114283,0.043920,0.083955,0.112561,0.046384,0.020477,0.003188,0.028484,0.004342,0.001033,0.007325,0.009963
Akita,0.354435,0.136861,0.032574,0.072441,0.120267,0.035701,0.059734,0.093902,0.036314,0.027523,0.003900,0.008732,0.002752,0.000637,0.005561,0.008664


In [178]:
eng_vote_totals.to_csv("../data/clean/election_results/house_of_councillors_20250720_vote_totals")
eng_vote_percent.to_csv("../data/clean/election_results/house_of_councillors_20250720_vote_percent")

In [184]:
#repeat largely the same steps for the next file
HoC_7_10_2022 = pd.read_excel("../data/raw/election_results/July 10 2022 House of Councillors election.xls", skiprows = 7)

In [185]:
HoC_7_10_2022.head()

,Unnamed: 0,Unnamed: 1,得票率,得票総数,を除く）の得票総数,Unnamed: 5,得票率.1,得票総数.1,を除く）の得票総数.1,Unnamed: 9,得票率.2,得票総数.2,を除く）の得票総数.2,Unnamed: 13,得票率.3,得票総数.3,を除く）の得票総数.3
0,北海道,"809,608.315",34.83,"623,864.785","185,743.530","500,331.595",21.53,"402,736.820","97,594.775","172,373.071",7.42,"151,597","20,776.071","273,557.726",11.77,"114,926","158,631.726"
1,青森県,"212,008.514",42.12,"162,983.666","49,024.848","94,348.481",18.75,"79,600.609","14,747.872","27,266.484",5.42,"25,404","1,862.484","52,363.291",10.40,"25,940","26,423.291"
2,岩手県,"205,311.741",37.60,"162,467.285","42,844.456","117,214.290",21.47,"97,251.870","19,962.420","31,408.455",5.75,"29,550","1,858.455","43,957.287",8.05,"24,219","19,738.287"
3,宮城県,"313,755.793",34.58,"238,761","74,994.793","173,700.855",19.14,"140,515.581","33,185.274","92,879.031",10.24,"86,487","6,392.031","98,694.245",10.88,"44,119","54,575.245"
4,秋田県,"197,585.770",45.33,"152,056","45,529.770","51,285.206",11.77,"38,478.672","12,806.534","48,101.728",11.04,"36,881","11,220.728","45,815.891",10.51,"23,517","22,298.891"


In [186]:
#only select columns for prefecture and 得票総数 for each party
vote_totals_only_2022 = HoC_7_10_2022.loc[:, ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 5', 'Unnamed: 9', 'Unnamed: 13']]

In [187]:
vote_totals_only_2022.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 5,Unnamed: 9,Unnamed: 13
0,北海道,"809,608.315","500,331.595","172,373.071","273,557.726"
1,青森県,"212,008.514","94,348.481","27,266.484","52,363.291"
2,岩手県,"205,311.741","117,214.290","31,408.455","43,957.287"
3,宮城県,"313,755.793","173,700.855","92,879.031","98,694.245"
4,秋田県,"197,585.770","51,285.206","48,101.728","45,815.891"


In [ ]:
page_1_2022 = vote_totals_only_2022.loc[:46, :]
page_2_2022 = vote_totals_only_2022.loc[54:100, :]
page_3_2022 = vote_totals_only_2022.loc[108:154, :]
page_4_2022 = vote_totals_only_2022.loc[162:208, :]

In [203]:
page_4_2022.tail()

,Unnamed: 0,Unnamed: 1,Unnamed: 5,Unnamed: 9,Unnamed: 13
204,熊本県,"1,026.983",694,432,"689,471.709"
205,大分県,851.166,447,257,"487,555.849"
206,宮崎県,505.493,404,608,"410,273.859"
207,鹿児島県,874.355,663,"1,159","628,865.733"
208,沖縄県,834.070,647,437,"552,453.540"


In [204]:
page_1_2022.columns = ['prefecture', '自由民主党', '立憲民主党', '日本維新の会', '公明党']
page_2_2022.columns = ['prefecture', '国民民主党', '日本共産党', 'れいわ新撰組', '社会民主党']
page_3_2022.columns = ['prefecture', 'NHK党', '参政党', '幸福実現党', 'ごぼうの党']
page_4_2022.columns = ['prefecture', '日本第一党', '新党くにもり', '維新政党・新風', 'total']


In [205]:
merged_vote_totals_2022 = pd.merge(page_1_2022, page_2_2022, on = 'prefecture')
merged_vote_totals_2022 = pd.merge(merged_vote_totals_2022, page_3_2022, on = 'prefecture')
merged_vote_totals_2022 = pd.merge(merged_vote_totals_2022, page_4_2022, on = 'prefecture')

In [207]:
merged_vote_totals_2022.tail()

,prefecture,自由民主党,立憲民主党,日本維新の会,公明党,国民民主党,日本共産党,れいわ新撰組,社会民主党,NHK党,参政党,幸福実現党,ごぼうの党,日本第一党,新党くにもり,維新政党・新風,total
42,熊本県,"302,467.202","65,753.366","76,507.202","99,181.586","23,302.732","26,127.853","24,060.215","18,192.727","16,704.023","30,080.536","2,390","2,551.284","1,026.983",694,432,"689,471.709"
43,大分県,"174,996.455","69,933.174","41,976.678","63,024.803","36,970.109","22,904.833","17,713.135","28,416.532","10,947.557","14,997.752","1,928","2,191.655",851.166,447,257,"487,555.849"
44,宮崎県,"161,815.831","59,350.933","30,598.859","60,751.460","24,785.389","16,944.695","13,260.185","16,691.613","8,743.440","13,323.624","1,300","1,190.337",505.493,404,608,"410,273.859"
45,鹿児島県,"269,834.685","86,874.124","55,867.075","76,441.298","28,515.204","23,945.339","21,674.369","18,964.636","16,151.498","23,933.805","1,912","2,055.345",874.355,663,"1,159","628,865.733"
46,沖縄県,"149,861.616","58,072.495","40,881.951","81,618.348","20,515.968","51,781.637","36,511.416","60,728.141","17,875.928","25,666.592","2,864","4,157.378",834.070,647,437,"552,453.540"


In [217]:
merged_vote_totals_2022 = merged_vote_totals_2022.replace(',', '', regex = True)

In [218]:
for column in merged_vote_totals_2022.columns[1:]:
    merged_vote_totals_2022[[column]] = merged_vote_totals_2022[[column]].astype(float)

In [219]:
merged_vote_totals_2022.head(2)

,prefecture,自由民主党,立憲民主党,日本維新の会,公明党,国民民主党,日本共産党,れいわ新撰組,社会民主党,NHK党,参政党,幸福実現党,ごぼうの党,日本第一党,新党くにもり,維新政党・新風,total
0,北海道,809608.315,500331.595,172373.071,273557.726,90460.200,189624.992,97375.982,36305.163,58707.218,68632.363,5357.0,7907.162,4347.527,7803.0,1901.0,2324292.314
1,青森県,212008.514,94348.481,27266.484,52363.291,22480.338,29974.321,18640.337,14311.830,14054.686,11585.910,1440.0,2133.212,872.403,785.0,1024.0,503288.807


In [222]:
merged_vote_percentage_2022 = merged_vote_totals_2022.iloc[:, 0:17].copy()
merged_vote_percentage_2022.iloc[:, 1:17] = merged_vote_percentage_2022.iloc[:, 1:17].div(merged_vote_totals_2022['total'], axis = 0)

In [224]:
merged_vote_percentage_2022.head(2)

,prefecture,自由民主党,立憲民主党,日本維新の会,公明党,国民民主党,日本共産党,れいわ新撰組,社会民主党,NHK党,参政党,幸福実現党,ごぼうの党,日本第一党,新党くにもり,維新政党・新風,total
0,北海道,0.348325,0.215262,0.074162,0.117695,0.038919,0.081584,0.041895,0.015620,0.025258,0.029528,0.002305,0.003402,0.001870,0.003357,0.000818,1.0
1,青森県,0.421246,0.187464,0.054177,0.104042,0.044667,0.059557,0.037037,0.028437,0.027926,0.023020,0.002861,0.004239,0.001733,0.001560,0.002035,1.0


In [225]:
merged_vote_totals_2022.set_index('prefecture', inplace = True)
merged_vote_percentage_2022.set_index('prefecture', inplace = True)

In [230]:
merged_vote_totals_2022.tail()

,自由民主党,立憲民主党,日本維新の会,公明党,国民民主党,日本共産党,れいわ新撰組,社会民主党,NHK党,参政党,幸福実現党,ごぼうの党,日本第一党,新党くにもり,維新政党・新風,total
prefecture,,,,,,,,,,,,,,,,
熊本県,302467.202,65753.366,76507.202,99181.586,23302.732,26127.853,24060.215,18192.727,16704.023,30080.536,2390.0,2551.284,1026.983,694.0,432.0,689471.709
大分県,174996.455,69933.174,41976.678,63024.803,36970.109,22904.833,17713.135,28416.532,10947.557,14997.752,1928.0,2191.655,851.166,447.0,257.0,487555.849
宮崎県,161815.831,59350.933,30598.859,60751.460,24785.389,16944.695,13260.185,16691.613,8743.440,13323.624,1300.0,1190.337,505.493,404.0,608.0,410273.859
鹿児島県,269834.685,86874.124,55867.075,76441.298,28515.204,23945.339,21674.369,18964.636,16151.498,23933.805,1912.0,2055.345,874.355,663.0,1159.0,628865.733
沖縄県,149861.616,58072.495,40881.951,81618.348,20515.968,51781.637,36511.416,60728.141,17875.928,25666.592,2864.0,4157.378,834.070,647.0,437.0,552453.540


In [231]:
eng_vote_totals_2022 = merged_vote_totals_2022.copy()
eng_vote_percent_2022 = merged_vote_percentage_2022.copy()

In [242]:
eng_vote_totals_2022 = eng_vote_totals_2022.rename(index=jp_prefect_to_eng)
eng_vote_totals_2022 = eng_vote_totals_2022.rename(columns=jp_to_eng)

eng_vote_percent_2022 = eng_vote_percent_2022.rename(index=jp_prefect_to_eng)
eng_vote_percent_2022 = eng_vote_percent_2022.rename(columns=jp_to_eng)

In [245]:
eng_vote_percent_2022.tail()

,Liberal Democratic Party,Constitutional Democratic Party,Japan Innovation Party,Komeito,Democratic Party For the People,Japanese Communist Party,Reiwa Shinsengumi,Social Democratic Party,NHK Party,Sanseito,The Happiness Realization Party,Gobonoto,Japan First Party,Shinto Kunimori,Ishin Party Shimpu,total
prefecture,,,,,,,,,,,,,,,,
Kumamoto,0.438694,0.095368,0.110965,0.143852,0.033798,0.037895,0.034897,0.026386,0.024227,0.043628,0.003466,0.003700,0.001490,0.001007,0.000627,1.0
Oita,0.358926,0.143436,0.086096,0.129267,0.075827,0.046979,0.036330,0.058284,0.022454,0.030761,0.003954,0.004495,0.001746,0.000917,0.000527,1.0
Miyazaki,0.394409,0.144662,0.074582,0.148075,0.060412,0.041301,0.032320,0.040684,0.021311,0.032475,0.003169,0.002901,0.001232,0.000985,0.001482,1.0
Kagoshima,0.429082,0.138144,0.088838,0.121554,0.045344,0.038077,0.034466,0.030157,0.025684,0.038059,0.003040,0.003268,0.001390,0.001054,0.001843,1.0
Okinawa,0.271266,0.105117,0.074001,0.147738,0.037136,0.093730,0.066090,0.109924,0.032357,0.046459,0.005184,0.007525,0.001510,0.001171,0.000791,1.0


In [243]:
eng_vote_totals_2022.columns

Index(['Liberal Democratic Party', 'Constitutional Democratic Party',
       'Japan Innovation Party', 'Komeito', 'Democratic Party For the People',
       'Japanese Communist Party', 'Reiwa Shinsengumi',
       'Social Democratic Party', 'NHK Party', 'Sanseito',
       'The Happiness Realization Party', 'Gobonoto', 'Japan First Party',
       'Shinto Kunimori', 'Ishin Party Shimpu', 'total'],
      dtype='object')

In [246]:
eng_vote_totals_2022.to_csv("../data/clean/election_results/house_of_councillors_20220710_vote_totals")
eng_vote_percent_2022.to_csv("../data/clean/election_results/house_of_councillors_20220710_vote_percent")